# Possession Lens — submission notebook

**Team:** Visca el Barça, visca Catalunya · **Frozen model:** 053

This notebook checks the submission assets, explains external evidence, optionally regenerates the CSV locally, and runs tests. It never trains on the provided game or scores its ground truth. All optional inference starts **off**.

The website is `dashboard.html` in the project root. The full experimental notebook is preserved at [Research history](archive/Research_History_20261116.ipynb). Later experimental models failed acceptance and have not replaced053.

In [ ]:
from pathlib import Path
import sys, json, subprocess, hashlib, importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "controlswitch").is_dir()), None)
assert ROOT is not None, "Open this notebook inside the Possession Lens project."
sys.path.insert(0, str(ROOT))
assert sys.version_info[:2] == (3, 12), "Select the Possession Lens Python3.12 GRU kernel."
RUN_INFERENCE = False
RUN_TESTS = True
DATA_ROOT = ROOT / "data/raw/MORE THAN A HACK"  # Set your licensed data directory before enabling inference.
OUTPUT = ROOT / "local/notebook-final-053"  # Choose a NEW directory for each run.
TEAM_CSV = "Visca el Barça, visca Catalunya.csv"
CONFIG = ROOT / "reports/external/short-spell-053-config.json"
EXPECTED_SHA256 = "2f5a445d2c3c10b3126b5bda3a5b9b11050d9a87410f19b7cbc5aaa88b928eeb"
print("Project:", ROOT.name, "| Python:", sys.version.split()[0])
print("Inference enabled:", RUN_INFERENCE, "| No training or challenge scoring in this notebook")

## 1. Check the pinned environment and model

CPU inference needs Python3.12 and `requirements-gru-inference-lock.txt`. For JupyterLab, install `requirements-gru-notebook.txt`. These checks load code and weights only, not match data.

In [ ]:
from controlswitch.recurrent_pipeline import load_pipeline
versions = {name: importlib.metadata.version(name) for name in ("numpy", "pandas", "torch", "ipykernel")}
display(pd.DataFrame([versions]))
bundle = load_pipeline(CONFIG, ROOT)
print("Model and nested asset hashes verified:", bundle["config"]["candidate"])
del bundle

## 2. External evidence and limits

The headline score is mean official possession-spell F1 over two heavily reused Bundesliga games under four tracking conditions. It is **not a challenge score or independent-match estimate**. The separate partial-provider check regressed; generalisation remains a limitation.

In [ ]:
study = json.loads((ROOT / "reports/external/short-spell-053-report.json").read_text())
selected = study["selected"]["scores"]
conditions = ("clean", "masked", "long_gap", "tracker_noise")
display(pd.DataFrame([{"Tracking condition": c, "Spell F1": selected[c]["spell_f1_mean"]} for c in conditions]))
print("Mean:", round(selected["mean_spell_f1"], 6), "| Reused external development data")
proof_path = ROOT / "reports/external/final-reproduction-20261116.json"
if proof_path.exists():
    proof = json.loads(proof_path.read_text())
    print("Latest source reproduced frozen CSV exactly:", proof["exact_match_to_frozen_csv"])
    print("Official format check:", proof["official_format_check_passed"], "| Challenge accuracy measured:", proof["challenge_accuracy_measured"])

## 3. Optional: regenerate the submission locally

Set `DATA_ROOT` to your licensed organiser files and choose a new `OUTPUT` directory. Then change `RUN_INFERENCE` to `True` and rerun the setup and this cell. No fitting, video inspection or ground-truth evaluation occurs. Existing outputs are never overwritten.

The CSV has199,319 data rows, H/A/D inside the published halves and X elsewhere. Companion files contain model estimates, hashes and observation-quality flags; flags are not calibrated confidence or verified events.

In [ ]:
if RUN_INFERENCE:
    from controlswitch.recurrent_pipeline import export
    result = export(DATA_ROOT, OUTPUT, CONFIG, TEAM_CSV)
    csv_path = OUTPUT / TEAM_CSV
    actual = hashlib.sha256(csv_path.read_bytes()).hexdigest()
    assert actual == EXPECTED_SHA256, "CSV differs from the frozen file; investigate without editing labels."
    check = subprocess.run([sys.executable, str(ROOT / "evaluation/official/evaluation/validate_submission.py"), str(csv_path)], cwd=ROOT, capture_output=True, text=True)
    assert check.returncode == 0, check.stdout + check.stderr
    print("199,319-row CSV regenerated and format checked. SHA-256:", actual)
    print("Saved:", csv_path)
else:
    print("Inference is off. No organiser data was read.")

## 4. Why later recovery models were not submitted

The local recovery expert found more short possessions but added false switches. Training-only calibration reduced some false positives but lost genuine recovered spells. The phase pilot used **external provider-event windows, not Metrica automatic tags**; it also lost recoveries. All remain research. The organisers' automatic tag file was audited locally for schema/range consistency without comparing it against challenge predictions, video or labels.

In [ ]:
rows = [{"Method": "Submitted 053", "Spell F1": selected["mean_spell_f1"], "Status": "Frozen submission"}]
for filename, key, title in [("short-proposals-070-report.json", "receipt_supported", "Bounded recovery070"), ("short-agreement-071-report.json", "receipt_supported", "Agreement recovery071")]:
    p = ROOT / "reports/external" / filename
    if p.exists():
        r = json.loads(p.read_text()); t = next(t for t in r["trials"] if t["name"] == key)
        rows.append({"Method": title, "Spell F1": t["scores"]["mean_spell_f1"], "Status": "Rejected"})
p = ROOT / "reports/external/candidate-calibration-073-report.json"
if p.exists():
    r = json.loads(p.read_text())
    rows.append({"Method": "Calibrated recovery073", "Spell F1": r["candidate"]["mean_spell_f1"], "Status": "Rejected"})
    print("Calibration retained", r["retention"]["retained_recovered"], "of", r["retention"]["prototype_recovered"], "prototype recoveries.")
p = ROOT / "reports/external/episode-phase-075-report.json"
if p.exists():
    r = json.loads(p.read_text())
    for t in r["trials"]:
        rows.append({"Method": "External phase pilot: " + t["name"], "Spell F1": t["scores"]["mean_spell_f1"], "Status": "Not transfer-validated; not submitted"})
display(pd.DataFrame(rows))

## 5. Run the checks

The tests use synthetic/public external fixtures and frozen model assets. Private research checks skip when their caches are absent. They do not validate predictions against the challenge video or use challenge labels for tuning.

In [ ]:
if RUN_TESTS:
    check = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-q"], cwd=ROOT, capture_output=True, text=True)
    if check.returncode:
        print(check.stdout + check.stderr)
        raise RuntimeError("Project tests failed.")
    print("Project tests passed.")
    print("\n".join((check.stdout + check.stderr).splitlines()[-4:]))

## Submission checklist

- Use the team-named CSV with the expected SHA-256 and the matching technical document.
- Use a public code repository; the development repository remains private until reviewed public release.
- Keep licensed inputs, frame-level tracking and private caches out of the repository.
- See [Technical document](../docs/submission/Visca%20el%20Bar%C3%A7a,%20visca%20Catalunya.md), [Reproduction](../docs/REPRODUCIBILITY.md), and [Release status](../docs/submission/RELEASE_STATUS.md).

Official deadline:17 November2026. No competition score or rank is established.